In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_7.py ──
"""
Shared infrastructure for Exercise 7 — Transfer Learning.

Contains: CIFAR-10 data loading, feature visualisation helpers,
ExperimentTracker/ModelRegistry setup, training harness.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex7_transfer_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — CIFAR-10 (full 50K, resized for ResNet-18)
# ════════════════════════════════════════════════════════════════════════
# ResNet-18 was designed for 224x224 ImageNet images. CIFAR-10 is 32x32.
# We resize to 96x96: ResNet's strided convolutions shrink spatial dims
# by 32x, so 32x32 would collapse to 1x1 before final pooling. 96x96
# gives a 3x3 final feature map — enough spatial information to learn.

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SIZE = 96
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Standard transforms: ImageNet normalisation + resize for ResNet
train_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.RandomHorizontalFlip(),
        T.RandomCrop(INPUT_SIZE, padding=8),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
val_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_cifar10() -> tuple[
    torchvision.datasets.CIFAR10,
    torchvision.datasets.CIFAR10,
    DataLoader,
    DataLoader,
]:
    """Load CIFAR-10 with ImageNet-style transforms.

    Returns:
        train_set, val_set, train_loader, val_loader
    """
    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=train_transform,
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=val_transform,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=0)

    print(
        f"CIFAR-10 (full): train={len(train_set)}, val={len(val_set)}, "
        f"classes={N_CLASSES}"
    )
    print(f"  Input size: {INPUT_SIZE}x{INPUT_SIZE} (resized for ResNet-18)")
    print(f"  Classes: {CLASS_NAMES}")

    return train_set, val_set, train_loader, val_loader


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transfer.db"
    registry_db = "sqlite:///mlfp05_transfer_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_transfer_learning", registry, True


def init_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS — shared by all technique files
# ════════════════════════════════════════════════════════════════════════


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Train a model and log everything to ExperimentTracker.

    Returns:
        train_losses, val_accs, train_accs (per-epoch)
    """
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"\n-- {name} --  trainable params: {n_trainable:,} / {n_total:,}")

    opt = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    train_losses: list[float] = []
    val_accs: list[float] = []
    train_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "trainable_params": str(n_trainable),
                "total_params": str(n_total),
                "epochs": str(epochs),
                "lr": str(lr),
                "batch_size": str(tr_loader.batch_size),
                "dataset_size": str(len(tr_loader.dataset)),
            }
        )

        for epoch in range(epochs):
            # -- Training --
            model.train()
            batch_losses = []
            correct = total = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
                correct += int((logits.argmax(dim=-1) == yb).sum().item())
                total += int(yb.size(0))
            train_losses.append(float(np.mean(batch_losses)))
            train_accs.append(correct / total)
            scheduler.step()

            # -- Validation --
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for xb, yb in vl_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
            val_accs.append(correct / total)

            await run.log_metrics(
                {
                    "train_loss": train_losses[-1],
                    "train_acc": train_accs[-1],
                    "val_acc": val_accs[-1],
                },
                step=epoch + 1,
            )

            print(
                f"  epoch {epoch + 1}/{epochs}  "
                f"loss={train_losses[-1]:.4f}  "
                f"train_acc={train_accs[-1]:.3f}  "
                f"val_acc={val_accs[-1]:.3f}"
            )

        await run.log_metrics(
            {
                "final_val_acc": val_accs[-1],
                "best_val_acc": max(val_accs),
                "final_train_loss": train_losses[-1],
            }
        )

    return train_losses, val_accs, train_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            tr_loader,
            vl_loader,
            epochs,
            lr,
        )
    )


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Register a trained model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="val_acc", value=val_acc),
            MetricSpec(name="final_loss", value=final_loss),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={val_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, val_acc, final_loss))


# ════════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION & VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════


def extract_features(
    model: nn.Module,
    loader: DataLoader,
    max_samples: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """Extract features from the penultimate layer (before fc head).

    Works for both ResNet (avgpool hook) and Sequential models.
    """
    model.eval()
    hook_features: list[torch.Tensor] = []
    labels: list[np.ndarray] = []

    def hook_fn(module, inp, out):
        del module, inp  # PyTorch hook protocol args; not used here
        hook_features.append(out.flatten(1).detach().cpu())

    # ResNet: hook into avgpool; Sequential: second-to-last layer
    if hasattr(model, "avgpool"):
        avgpool = model.avgpool
        assert isinstance(
            avgpool, nn.Module
        ), f"expected nn.Module for avgpool, got {type(avgpool).__name__}"
        handle = avgpool.register_forward_hook(hook_fn)
    else:
        assert isinstance(
            model, nn.Sequential
        ), f"hook fallback requires nn.Sequential, got {type(model).__name__}"
        handle = model[-3].register_forward_hook(hook_fn)

    with torch.no_grad():
        collected = 0
        for xb, yb in loader:
            if collected >= max_samples:
                break
            xb = xb.to(device)
            model(xb)
            labels.append(yb.numpy())
            collected += len(yb)

    handle.remove()
    features_np = torch.cat(hook_features, dim=0).numpy()[:max_samples]
    labels_np = np.concatenate(labels)[:max_samples]
    return features_np, labels_np


def compute_tsne(features: np.ndarray, perplexity: int = 30) -> np.ndarray:
    """Run t-SNE dimensionality reduction to 2D."""
    # sklearn 1.5+ renamed n_iter → max_iter in TSNE.__init__
    tsne = TSNE(n_components=2, perplexity=perplexity, max_iter=500, random_state=42)
    return tsne.fit_transform(features)


def cluster_quality(coords: np.ndarray, labels: np.ndarray) -> float:
    """Cluster quality: ratio of intra-class to inter-class distance (lower = better)."""
    intra = []
    centroids = []
    for c in range(N_CLASSES):
        mask = labels == c
        if mask.sum() < 2:
            continue
        pts = coords[mask]
        centroid = pts.mean(axis=0)
        centroids.append(centroid)
        intra.append(np.mean(np.linalg.norm(pts - centroid, axis=1)))
    centroids_arr = np.array(centroids)
    inter = np.mean(
        [
            np.linalg.norm(centroids_arr[i] - centroids_arr[j])
            for i in range(len(centroids_arr))
            for j in range(i + 1, len(centroids_arr))
        ]
    )
    avg_intra = np.mean(intra)
    return float(avg_intra / inter) if inter > 0 else float("inf")


def plot_tsne(
    coords: np.ndarray,
    labels: np.ndarray,
    title: str,
    output_path: str | Path,
) -> None:
    """Create and save a t-SNE scatter plot coloured by class."""
    fig = go.Figure()
    for c in range(N_CLASSES):
        mask = labels == c
        fig.add_trace(
            go.Scatter(
                x=coords[mask, 0],
                y=coords[mask, 1],
                mode="markers",
                name=CLASS_NAMES[c],
                marker=dict(size=4, opacity=0.6),
            )
        )
    fig.update_layout(
        title=title,
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        template="plotly_white",
    )
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance."""
    return ModelVisualizer()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(metrics=metrics, x_label="Epoch", y_label="Value")
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def count_params(model: nn.Module, trainable_only: bool = False) -> int:
    """Count parameters in a model."""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 7, Part 4: Adapter Modules (Bridge to M6 LoRA)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Understand why full fine-tuning is expensive and risky
#   - Build bottleneck adapter modules that inject small trainable
#     layers inside a frozen backbone
#   - Compare parameter efficiency across methods: from-scratch,
#     frozen head, adapter, and (preview) LoRA
#   - Visualise the performance-vs-parameters Pareto frontier
#   - Apply adapter concepts to a multi-tenant AI platform scenario
#
# PREREQUISITES: Parts 1-3 (baseline, transfer, data efficiency).
# ESTIMATED TIME: ~25 min
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision



## THEORY — The Parameter Efficiency Spectrum


In [ ]:
# Transfer learning has a spectrum of approaches, each trading off
# between trainable parameters and model capacity:
#
# FROZEN HEAD (Part 2):
#   - Update ONLY the final classifier layer (~0.05% of params)
#   - Fastest training, zero risk of "catastrophic forgetting"
#   - Limited: the backbone features might not be perfectly suited
#
# FULL FINE-TUNING:
#   - Update ALL parameters (100% trainable)
#   - Highest potential accuracy
#   - Risk: "catastrophic forgetting" — the model unlearns ImageNet
#     features while overfitting to your small dataset
#   - Storage: need a full copy of the model per task
#
# ADAPTER MODULES (this section):
#   - Inject small trainable bottleneck layers INSIDE the frozen backbone
#   - ~5-10% of params trainable — good balance
#   - Skip connection: adapter starts as identity, so training begins
#     from the pre-trained features
#   - Storage: only save the adapter weights (~1-5 MB) per task
#
# LoRA (Module 6):
#   - Low-rank decomposition: W + A @ B where A is d x r, B is r x d
#   - ~1-5% of params trainable — best for LLMs with billions of params
#   - Same idea as adapters but uses matrix decomposition instead of
#     bottleneck layers
#
# The key insight: you don't need to update every parameter to adapt a
# model. Most of the pre-trained knowledge is useful as-is. You only
# need to nudge the model slightly toward your specific task.



In [ ]:
print("\n" + "=" * 70)
print("  PART 4: Adapter Modules (Bridge to M6 LoRA)")
print("=" * 70)



## TASK 1 — Load data and set up engines


In [ ]:
train_set, val_set, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()



## TASK 2 — Build the BottleneckAdapter module


The skip connection ensures the adapter starts as an identity function
    (initial up-project weights are zero), so training begins from the
    pre-trained features — not random noise.

    Architecture:
        x -> x + up_project(ReLU(down_project(x)))

    Args:
        dim: Input/output dimension (must match the layer it's inserted into)
        bottleneck: Bottleneck dimension (smaller = fewer params, less capacity)


In [ ]:
# The adapter is a small trainable module: down-project to a bottleneck
# dimension, apply a nonlinearity, then up-project back. The SKIP
# CONNECTION is critical: it means the adapter starts as an identity
# function (because up-project weights are initialised to zero), so
# training begins from the pre-trained features, not random noise.


class BottleneckAdapter(nn.Module):

    def __init__(self, dim: int, bottleneck: int = 64):
        super().__init__()
        # TODO: Create two linear layers and zero-init the up-projection
        # Steps:
        #   1. self.down = nn.Linear(dim, bottleneck)
        #   2. self.up = nn.Linear(bottleneck, dim)
        #   3. nn.init.zeros_(self.up.weight)
        #   4. nn.init.zeros_(self.up.bias)
        # Why zero-init? So the adapter starts as identity (output = input)
        ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement skip connection: x + up(relu(down(x)))
        # Hint: return x + self.up(F.relu(self.down(x)))
        ____



### Checkpoint 1


In [ ]:
adapter_test = BottleneckAdapter(dim=256, bottleneck=64)
test_input = torch.randn(2, 256)
test_output = adapter_test(test_input)
# At init, the adapter is identity (zero up-projection)
assert torch.allclose(
    test_input, test_output, atol=1e-6
), "Adapter should start as identity function (zero-init up-projection)"
adapter_params = sum(p.numel() for p in adapter_test.parameters())
expected_params = 256 * 64 + 64 + 64 * 256 + 256  # down + bias + up + bias
assert (
    adapter_params == expected_params
), f"Adapter params: {adapter_params} vs expected {expected_params}"
# INTERPRETATION: The zero-init trick is crucial. Without it, the
# adapter would add random noise to the pre-trained features on the
# first forward pass, destroying the valuable ImageNet representations.
# With zero-init, training starts from "no change" and gradually learns
# task-specific adjustments.
print(f"  BottleneckAdapter(256, 64): {adapter_params:,} params, starts as identity")
print("--- Checkpoint 1 passed --- adapter module verified\n")



## TASK 3 — Build adapter-augmented ResNet-18


Wraps a ResNet block with a channel-wise adapter.


ResNet-18 with bottleneck adapters after layer3 and layer4.


In [ ]:
# We insert adapters after ResNet's layer3 and layer4 blocks. The
# adapter operates on channel-wise pooled features (spatial average
# pooling to get a vector per sample, then adapter, then broadcast
# back to spatial dimensions).


class AdaptedBlock(nn.Module):

    def __init__(self, block: nn.Module, adapter: BottleneckAdapter):
        super().__init__()
        self.block = block
        self.adapter = adapter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Run block, then apply adapter on channel-wise pooled features
        # Steps:
        #   1. out = self.block(x)
        #   2. b, c, h, w = out.shape
        #   3. pooled = out.mean(dim=[2, 3])  -> (B, C)
        #   4. adapted = self.adapter(pooled)  -> (B, C)
        #   5. return out + adapted.unsqueeze(-1).unsqueeze(-1)
        # Hint: unsqueeze broadcasts (B,C) back to (B,C,1,1) for addition
        ____


def build_adapter_resnet(
    n_classes: int = N_CLASSES,
    bottleneck: int = 64,
) -> nn.Module:
    # TODO: Build adapter-augmented ResNet-18
    # Steps:
    #   1. Load pre-trained ResNet-18 (same as Part 2)
    #   2. Freeze all parameters
    #   3. Replace model.fc with nn.Linear(in_features, n_classes)
    #   4. Wrap model.layer3 with AdaptedBlock(original_layer3, BottleneckAdapter(256, bottleneck))
    #   5. Wrap model.layer4 with AdaptedBlock(original_layer4, BottleneckAdapter(512, bottleneck))
    # Hint: Save original layers before replacing:
    #   original_layer3 = model.layer3
    #   model.layer3 = AdaptedBlock(original_layer3, BottleneckAdapter(256, bottleneck))
    ____


adapter_model = build_adapter_resnet(bottleneck=64)
adapter_model.to(device)
n_adapter_trainable = count_params(adapter_model, trainable_only=True)
n_adapter_total = count_params(adapter_model)

print(
    f"  Adapter ResNet-18: {n_adapter_trainable:,} trainable / "
    f"{n_adapter_total:,} total "
    f"({100 * n_adapter_trainable / n_adapter_total:.1f}%)"
)



### Checkpoint 2


In [ ]:
assert n_adapter_trainable > 5000, "Adapter should have more params than frozen head"
assert n_adapter_trainable < n_adapter_total, "Should have fewer trainable than total"
# INTERPRETATION: The adapter adds ~100K trainable parameters on top of
# the ~5K frozen-head params. This is still far fewer than the ~11M
# total, but gives the model more capacity to adapt to the task.
print("--- Checkpoint 2 passed --- adapter ResNet built\n")



## TASK 4 — Train and compare all three methods


In [ ]:
print("\n" + "=" * 70)
print("  TRAINING: All three approaches on CIFAR-10")
print("=" * 70)

# TODO: Train all three models and collect results
# Method 1: Adapter model (already built above)
# Method 2: Frozen head (build_frozen_head function)
# Method 3: From scratch (build_scratch_cnn function)
# Hint: Use train_model() from helpers for each
# Hint: train_model returns (losses, val_accs, train_accs)

# Method 1: Adapter model
adapter_losses, adapter_accs, _ = train_model(
    adapter_model,
    "adapter_resnet18",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_adapter = max(adapter_accs)


# Method 2: Frozen head (from Part 2)
def build_frozen_head(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Same as Part 2 — load ResNet-18, freeze, replace fc
    ____


frozen_model = build_frozen_head()
frozen_losses, frozen_accs, _ = train_model(
    frozen_model,
    "frozen_head_resnet18",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_frozen = max(frozen_accs)
n_frozen_trainable = count_params(frozen_model, trainable_only=True)


# Method 3: From scratch
def build_scratch_cnn(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Same 3-layer CNN from Part 1
    ____


scratch_model = build_scratch_cnn()
scratch_losses, scratch_accs, _ = train_model(
    scratch_model,
    "cnn_from_scratch",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)
best_scratch = max(scratch_accs)
n_scratch_trainable = count_params(scratch_model, trainable_only=True)



### Checkpoint 3


In [ ]:
assert (
    best_adapter > 0.20
), f"Adapter model should beat random chance (acc={best_adapter:.3f})"
assert (
    n_adapter_trainable < n_adapter_total
), "Adapter model should have fewer trainable params than total"
print(f"\n  === Parameter Efficiency Comparison ===")
print(f"  {'Method':<25} {'Trainable':>12} {'Val Acc':>10}")
print("  " + "-" * 50)
print(f"  {'From scratch':<25} {n_scratch_trainable:>12,} {best_scratch:>10.4f}")
print(f"  {'Frozen head':<25} {n_frozen_trainable:>12,} {best_frozen:>10.4f}")
print(
    f"  {'Adapter (bottleneck)':<25} {n_adapter_trainable:>12,} {best_adapter:>10.4f}"
)
print("\n--- Checkpoint 3 passed --- all three methods compared\n")



## TASK 5 — Visualise: Parameter count vs performance Pareto chart


In [ ]:
# TODO: Create two visualisations:
# 1. Pareto chart: trainable params (x, log scale) vs accuracy (y)
#    - Scatter with 3 methods + LoRA preview point
#    - LoRA estimate: ~2% of total params, ~98% of adapter accuracy
# 2. Training curves: epoch vs accuracy for all three methods
# Hint: fig_pareto = go.Figure()
# Hint: fig_pareto.add_trace(go.Scatter(x=param_counts, y=accuracies, mode="markers+text"))
# Hint: fig_pareto.update_layout(xaxis_type="log")

methods = ["From Scratch", "Frozen Head", "Adapter (bottleneck=64)"]
param_counts = [n_scratch_trainable, n_frozen_trainable, n_adapter_trainable]
accuracies = [best_scratch, best_frozen, best_adapter]
colours = ["#FF5722", "#2196F3", "#4CAF50"]

fig_pareto = go.Figure()
____

pareto_path = OUTPUT_DIR / "04_parameter_pareto.html"
fig_pareto.write_html(str(pareto_path))
print(f"  Saved: {pareto_path}")

# Training curves comparison
fig_curves = go.Figure()
epochs_x = list(range(1, EPOCHS + 1))
____

curves_path = OUTPUT_DIR / "04_training_curves.html"
fig_curves.write_html(str(curves_path))
print(f"  Saved: {curves_path}")



### Checkpoint 4


In [ ]:
assert pareto_path.exists(), "Pareto chart should be saved"
assert curves_path.exists(), "Training curves should be saved"
print("--- Checkpoint 4 passed --- visualisations complete\n")



## TASK 6 — Apply: Multi-Tenant AI Platform in Singapore


In [ ]:
# SCENARIO: You're building an AI platform in Singapore that serves
# multiple clients. Each client needs a custom image classifier, but
# they share the same base architecture (ResNet-18).
#
# Full fine-tuning: store 11M params per client = ~44 MB per model
# Adapter approach: store ~100K params per client = ~0.4 MB per adapter
#
# For 50 clients, that's 2.2 GB vs 20 MB. And at inference time, you
# can keep ONE ResNet-18 in GPU memory and swap adapters per request.

print("\n" + "=" * 70)
print("  APPLY: Multi-Tenant AI Platform — One Base, Many Adapters")
print("=" * 70)

N_CLIENTS = 50
FULL_MODEL_MB = n_adapter_total * 4 / (1024 * 1024)  # float32, 4 bytes each
ADAPTER_MB = n_adapter_trainable * 4 / (1024 * 1024)

# TODO: Print the multi-tenant storage analysis
# Steps:
#   1. Print per-client and 50-client storage for full fine-tuning
#   2. Print per-client and 50-client storage for adapter approach
#      (adapter: 1 base model + N tiny adapters)
#   3. Calculate savings_storage and savings_gpu
#   4. Print inference cost comparison
# Hint: Full = N_CLIENTS * FULL_MODEL_MB
# Hint: Adapter = FULL_MODEL_MB + N_CLIENTS * ADAPTER_MB (one shared base)
# Hint: GPU at inference: full needs N models, adapter needs 1 base + swap
print(f"\n  === Multi-Tenant Storage Analysis (50 clients) ===")
print(f"  Base model (ResNet-18): {n_adapter_total:,} params = {FULL_MODEL_MB:.1f} MB")
print(f"  Adapter weights: {n_adapter_trainable:,} params = {ADAPTER_MB:.2f} MB")
____

savings_storage = N_CLIENTS * FULL_MODEL_MB - (FULL_MODEL_MB + N_CLIENTS * ADAPTER_MB)
savings_gpu = N_CLIENTS * FULL_MODEL_MB - (FULL_MODEL_MB + ADAPTER_MB)

print(
    f"\n  Storage savings: {savings_storage:.0f} MB ({savings_storage / (N_CLIENTS * FULL_MODEL_MB) * 100:.0f}%)"
)
print(f"  GPU memory savings: {savings_gpu:.0f} MB (load one base + swap adapters)")
print()
print(f"  BRIDGE TO M6:")
print(f"  In M6, you will learn LoRA (Low-Rank Adaptation) — the adapter")
print(f"  technique designed for LLMs with BILLIONS of parameters.")
print(f"  Same concept: inject small trainable matrices into frozen weights.")
print(f"  LoRA uses low-rank decomposition: W + A @ B where rank r << d.")
print(f"  A GPT-scale model with 7B params might need only ~4M LoRA params (0.06%).")



### Checkpoint 5


In [ ]:
assert savings_storage > 0, "Adapter approach should save storage"
assert ADAPTER_MB < FULL_MODEL_MB, "Adapter should be smaller than full model"
# INTERPRETATION: Adapters make multi-tenant ML serving economically
# viable. Instead of N copies of the full model (one per client), you
# store one shared backbone plus N tiny adapter files. This is the
# same principle that makes LoRA practical for LLM fine-tuning in M6.
print("\n--- Checkpoint 5 passed --- multi-tenant analysis complete\n")



## CLEANUP


In [ ]:
await conn.close()



## REFLECTION


[x] Built a BottleneckAdapter module with zero-init skip connection
  [x] Inserted adapters into frozen ResNet-18 (layer3 + layer4)
  [x] Compared three approaches:
      From scratch: {n_scratch_trainable:>8,} params -> {best_scratch:.1%} accuracy
      Frozen head:  {n_frozen_trainable:>8,} params -> {best_frozen:.1%} accuracy
      Adapter:      {n_adapter_trainable:>8,} params -> {best_adapter:.1%} accuracy
  [x] Visualised the parameter-efficiency Pareto frontier
  [x] Analysed multi-tenant serving: {savings_storage:.0f} MB saved across 50 clients

  THE TRANSFER LEARNING SPECTRUM:
    Frozen head  ->  Adapter/LoRA  ->  Partial fine-tune  ->  Full fine-tune
    (fewest params)                                         (most params)
    (fastest training)                                      (highest capacity)
    (safest from forgetting)                                (risk of forgetting)

  BRIDGE TO M6 — LoRA:
    Adapters: bottleneck layers (down -> ReLU -> up)
    LoRA:     low-rank matrices (W + A @ B, where r << d)
    Same principle, different math. LoRA is preferred for LLMs because
    the low-rank decomposition is more parameter-efficient at scale.

  NEXT: Part 5 deploys the best model to production with ONNX export
  and InferenceServer — the full pipeline from experiment to serving.


In [ ]:
print("\n" + "=" * 70)
print("  PART 4 COMPLETE — What You've Learned")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # Frozen backbone + small adapter layers
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (Adapter modules — parameter-efficient fine-tuning) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        model,
        train_loader,
        _diag_loss,
        title="Adapter modules — parameter-efficient fine-tuning",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow (HEALTHY): Only adapter layers receive gradient —
#     RMS 2.3e-03 on adapter params, 0 on frozen backbone (expected).
# [✓] 0.5% of parameters trainable, 85% val accuracy — same as full fine-tune.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [BLOOD TEST — ADAPTER-SPECIFIC] The "0 gradient on backbone"
#     is by design, not a bug. Diagnostic correctly shows frozen
#     layers as inactive. Only adapter bottlenecks receive gradient.
#
#  [PRESCRIPTION] Adapters = 200x fewer params to store per task.
#     For production deployment: one frozen backbone + many
#     per-task adapters. Training cost: fraction of full fine-tune.
#     Quality: typically within 1% of full fine-tune.
#     Slide 5.7 references this as the modern 2024+ approach
#     (HuggingFace PEFT library, LoRA).

